In [1]:
from bisect import bisect_left
from collections.abc import Callable
from functools import partial
from random import Random
from statistics import median
from timeit import repeat
from typing import Any

In [2]:
catalog = {101: "винт", 106: "шайба", 111: "гайка", 204: "болт", 315: "подшипник"}
requested_code = 315
bucket_count = 5
codes = [101, 106, 111, 204, 106, -4]
own_codes = [101, 106, 111, 204, -4]
own_query = 116
experiment_n = 2000
BENCHMARK_NUMBER = 5
BENCHMARK_REPEATS = 5

# ДЗ №4. Поиск и хеширование

Курс «Алгоритмы и структуры данных»

**Фамилия, группа:** [REDACTED NAME], [REDACTED GROUP]

## Как работать с этим ноутбуком

- Дважды щёлкните по первой ячейке, впишите фамилию и группу и нажмите Shift+Enter.
- Демо-ячейки уже выполнены. Запускайте их сверху вниз через Shift+Enter.
  Строки с пометкой `# <-- можно менять здесь` подходят для экспериментов.
- В ячейках «Ответ» замените курсивную подсказку своим текстом. В коде заполните места с пометкой `TODO`.
- Для запуска достаточно Python 3.9+ и стандартной библиотеки. В замерах используйте числа со своего компьютера.

## Шпаргалка

| Понятие | Как читать пример |
|---|---|
| Ключ | У детали это числовой код. По нему можно найти название в словаре. |
| Корзина | В учебной таблице номер корзины равен остатку от деления кода на число корзин. |
| Коллизия | Если разные коды попали в одну корзину, поиск проверяет коды в её цепочке по очереди. |

---

## Задача 1. Найдите деталь в каталоге

На складе четыре вида деталей. Покупатель называет код: если деталь
есть, покажем название; если нет — честно сообщим об отсутствии.
Запустите маленький каталог.

In [3]:
if requested_code in catalog:
    print("Нашли:", requested_code, "—", catalog[requested_code])
else:
    print("Кода", requested_code, "в каталоге нет")

for code in [101, 111, 116, requested_code]:
    print("Запрос", code, "→", catalog.get(code, "нет в каталоге"))

Нашли: 315 — подшипник
Запрос 101 → винт
Запрос 111 → гайка
Запрос 116 → нет в каталоге
Запрос 315 → подшипник


`catalog` — **словарь**, или `dict`: код слева от `:` — ключ,
название справа — связанное с ним значение. `106 in catalog` проверяет
наличие ключа; `catalog[106]` получает его название. `if` выбирает
ветку, `else` выполняется, когда условие ложно.

Попробуйте 101, 111 и отсутствующий 116. Затем добавьте одну пару
«код: название» в каталог и найдите её. Какой вывод ожидаете до запуска?

Готовый словарь уже умеет быстро искать. Дальше разберём одну из идей,
которая делает поиск быстрым. Нас пока интересует **только наличие кода**:
ответ `True` — есть, `False` — нет. Названия временно оставим за кадром.
Повторный код не создаёт новый вид детали: храним множество кодов.

### Ответ (задача 1)

Добавил код 315 со значением «подшипник», запросил 315 и получил эту запись.
До запуска ожидал, что 101 даст «винт», 111 — «гайка», 116 отсутствует, а новый код 315 будет найден.
Сохранённый вывод подтверждает эти ожидания. dict здесь нужен, потому что вместе с наличием хранится название.

### Разложим коды по корзинам

Пусть корзин пять, а адрес кода — остаток `code % 5`.
Например, `101 % 5 = 1`, `106 % 5 = 1`, `111 % 5 = 1`.
Разные коды получили один адрес: это **коллизия**.
Не выбрасываем предыдущие коды, а сохраняем в корзине список — цепочку.

Ячейка ниже — готовая учебная таблица. Она не воспроизводит внутренности
Python `dict`, а делает коллизии видимыми. Ключи здесь — целые числа,
число корзин — положительное целое. `class` объединяет данные и операции,
`self.buckets` — корзины конкретной таблицы. Создавать классы самому не нужно.
Метод `add` вычисляет корзину и добавляет код, только если его ещё нет.

In [4]:
class ChainedIntSet:
    """Учебное множество целых чисел: цепочки, без resize и удаления."""

    def __init__(self, capacity):
        if type(capacity) is not int or capacity <= 0:
            raise ValueError("capacity должна быть положительным целым")
        self.buckets = [[] for _ in range(capacity)]

    def add(self, value):
        bucket = self.buckets[value % len(self.buckets)]
        if value not in bucket:
            bucket.append(value)

**Сначала предскажите:** в какой корзине окажется 204? Что изменит
повтор 106? Затем выполните ячейку. Поменяйте число корзин с 5 на 7,
потом на 1: сразу видно, как меняется распределение.

In [5]:
demo = ChainedIntSet(bucket_count)
for value in codes:
    demo.add(value)
for address, bucket in enumerate(demo.buckets):
    print("Корзина", address, ":", bucket)

for alternative_capacity in (7, 1):
    alternative = ChainedIntSet(alternative_capacity)
    for value in codes:
        alternative.add(value)
    print("Корзин:", alternative_capacity, "распределение:", alternative.buckets)
    assert sum(map(len, alternative.buckets)) == len(set(codes))

Корзина 0 : []
Корзина 1 : [101, 106, 111, -4]
Корзина 2 : []
Корзина 3 : []
Корзина 4 : [204]
Корзин: 7 распределение: [[], [106, 204], [], [101, -4], [], [], [111]]
Корзин: 1 распределение: [[101, 106, 111, 204, -4]]


При пяти корзинах в корзине 1 лежат `[101, 106, 111, -4]`, а в корзине
4 — `[204]`. Повтор 106 ничего не добавил. Отрицательный учебный код
допустим: в Python `-4 % 5 = 1`. Это удобная проверка границ алгоритма,
даже если реальные коды вашего склада только положительные.

`for value in codes` по очереди берёт элементы списка.
`enumerate` выдаёт номер корзины и саму корзину. Все внутренние списки
создаются отдельно: иначе добавление в один могло бы изменить остальные.

### Неудачный вариант: непустая корзина значит «нашли»?

Проверим такую догадку на коде 116. У него тот же адрес 1, что у 101,
но это **другая деталь**. Вернём для контрпримера пять корзин независимо
от того, какое число вы попробовали выше.

In [6]:
example = ChainedIntSet(5)
for value in [101, 106, 111]:
    example.add(value)
target = 116
bucket = example.buckets[target % len(example.buckets)]
bad_answer = bool(bucket)  # ошибочно: спрашиваем лишь «не пусто ли?»
print("Кандидаты:", bucket)
print("Неудачный ответ:", bad_answer)
print("Правильный ответ:", target in [101, 106, 111])
assert bad_answer is True and target not in [101, 106, 111]

Кандидаты: [101, 106, 111]
Неудачный ответ: True
Правильный ответ: False


Номер корзины только сужает круг кандидатов. Теперь нужно проверить
**равенство самих кодов**. Для 106 сравним 101 → 106 и остановимся:
два сравнения. Для отсутствующего 116 просмотрим 101 → 106 → 111:
три сравнения. Пустая корзина сразу даёт отсутствие и ноль сравнений
кодов, хотя вычисление адреса всё равно требует работы.

---

## Задача 2. Реализуйте поиск внутри цепочки

Заполните `contains_count(table, target)`. Возвращается пара
`(найден ли код, число сравнений)`, например `(True, 2)`.
Это кортеж: первая позиция хранит логический ответ, вторая — счётчик.
Не меняйте таблицу и не просматривайте чужие корзины.

Корзина уже вычислена в первой строке. Дальше нужен небольшой цикл:
заведите счётчик, перебирайте кандидатов, учитывайте каждую проверку `==`;
при равенстве сразу возвращайте успех. После окончания цепочки верните
отсутствие. `return` завершает функцию; `comparisons += 1` увеличивает
счётчик. Готовые `in`, `set` и `.index()` внутри вашей функции замените
явным циклом: иначе механизм и число сравнений будут спрятаны.

### Ответ (задача 2)

Вычисляю единственную возможную корзину и проверяю её элементы явным циклом.
Перед каждой проверкой равенства увеличиваю счётчик; при совпадении сразу возвращаю True и число сравнений.
При исчерпании цепочки возвращаю False и счётчик, а для пустой корзины — (False, 0).
Другие корзины не нужны: при добавлении цель получила бы тот же остаток от деления.
Таблица при поиске не изменяется. Для цепочки длины k требуется не больше k сравнений и O(1) дополнительной памяти.

In [7]:
def contains_count(table: ChainedIntSet, target: int) -> tuple[bool, int]:
    bucket = table.buckets[target % len(table.buckets)]
    comparisons = 0
    for candidate in bucket:
        comparisons += 1
        if candidate == target:
            return True, comparisons
    return False, comparisons

Проверки уже написаны, их менять не нужно. Пустая функция возвращает
`None`, поэтому вывод будет **НЕ ВЫПОЛНЕНО**. После реализации проверятся
пустая корзина, повторные коды, коллизии, отрицательные значения и счётчики.
**ПРОЙДЕНО** относится к этим тестам, ваши выводы остаются отдельной частью.
В коде проверки `lambda: contains_count(...)` — короткая функция без
аргументов: проверяющая программа сама вызывает её в нужный момент.

In [8]:
CHECK_RESULTS = {}


def check_work(name, probe, tests):
    """None means unfinished; an exception means a failed check, not success."""
    try:
        if probe() is None:
            CHECK_RESULTS[name] = "НЕ ВЫПОЛНЕНО"
            print(name + ": НЕ ВЫПОЛНЕНО — заполните TODO.")
            return False
        tests()
    except Exception as error:
        CHECK_RESULTS[name] = "ОШИБКА"
        print(name + ": ОШИБКА — " + type(error).__name__)
        print(str(error)[:160])
        return False
    CHECK_RESULTS[name] = "ПРОЙДЕНО"
    print(name + ": ПРОЙДЕНО — открытые тесты; объяснения отдельно.")
    return True


def test_hash_search():
    examples = [
        (5, [], 1, (False, 0)),
        (5, [2, 7, 12, 3, 7, -3], 7, (True, 2)),
        (5, [2, 7, 12, 3, 7, -3], 17, (False, 4)),
        (5, [2, 7, 12, 3, 7, -3], -3, (True, 4)),
        (5, [2, 7, 12, 3, 7, -3], 4, (False, 0)),
        (1, [0, 1, -1, 2, 2], 2, (True, 4)),
        (1, [0, 1, -1, 2, 2], 99, (False, 4)),
    ]
    for capacity, data, target, expected in examples:
        table = ChainedIntSet(capacity)
        for value in data:
            table.add(value)
        before = [bucket[:] for bucket in table.buckets]
        actual = contains_count(table, target)
        assert isinstance(actual, tuple) and len(actual) == 2
        assert type(actual[0]) is bool and type(actual[1]) is int
        assert actual == expected, (data, target, actual, expected)
        assert table.buckets == before, "Поиск изменил таблицу"
        assert sum(map(len, table.buckets)) == len(set(data))
    rng = Random(42)
    for capacity in [1, 2, 5, 17]:
        data = [rng.randrange(-30, 31) for _ in range(80)]
        table = ChainedIntSet(capacity)
        for value in data:
            table.add(value)
        for target in range(-40, 41):
            bucket = table.buckets[target % capacity]
            expected_count = bucket.index(target) + 1 if target in bucket else len(bucket)
            assert contains_count(table, target) == (target in set(data), expected_count)


hash_ok = check_work("Поиск в цепочке", lambda: contains_count(demo, 7), test_hash_search)

Поиск в цепочке: ПРОЙДЕНО — открытые тесты; объяснения отдельно.


### Почему весь каталог просматривать не нужно

При добавлении каждый ключ попал в корзину `key % m`. Если цель есть,
она обязательно в той же корзине. До очередного сравнения все уже
просмотренные кандидаты не подошли; конец цепочки доказывает отсутствие.
Так мы одновременно объясняем, почему поиск правильный и почему конечен.

O(...) обозначает верхнюю оценку того, как растёт число операций
с размером данных, а не время в миллисекундах.
Если n различных кодов распределены по m корзинам достаточно равномерно,
ожидаемая длина цепочки порядка n/m. Поиск стоит O(1+n/m), считая хеш
и сравнение небольших целых за постоянную работу. Но это условие:
при всех кодах в одной корзине худший поиск — O(n). В нашей таблице
корзины сами не добавляются, поэтому O(1) при любом росте базы не обещаем.

---

## Задача 3. Составьте свой каталог и проверьте запрос

Придумайте 4–6 различных кодов, среди которых есть хотя бы два с
одинаковым остатком от деления на 5. Выберите один присутствующий или
отсутствующий запрос. Впишите список и запрос вместо `None`.
Предскажите логический ответ и число сравнений до запуска.

### Ответ (задача 3)

Мои коды деталей: 101, 106, 111, 204 и −4; отрицательный код — учебный граничный случай.
Запрос — отсутствующий код 116. При пяти корзинах 101, 106, 111 и −4 имеют остаток 1,
как и 116, поэтому ожидаю (False, 4): нужно проверить всю эту цепочку.
Код 204 находится в корзине 4 и в поиске не участвует. Для демонстрационного набора повтор 106
не добавляет новый элемент. При одной корзине все различные коды оказываются в одной цепочке,
что показывает возможный линейный худший случай.

In [9]:
print("Свои коды:", own_codes, "запрос:", own_query)

Свои коды: [101, 106, 111, 204, -4] запрос: 116


In [10]:
if own_codes is None or own_query is None or not hash_ok:
    own_result = None
    print("Свой поиск: НЕ ВЫПОЛНЕНО — нужны функция, коды и запрос.")
else:
    own_table = ChainedIntSet(5)
    for value in own_codes:
        own_table.add(value)
    own_result = contains_count(own_table, own_query)
    found, comparisons = own_result
    assert found == (own_query in own_codes)
    print("Коды:", own_codes)
    print("Корзина запроса:", own_table.buckets[own_query % 5])
    print("Нашли:", found, "сравнений:", comparisons)

Коды: [101, 106, 111, 204, -4]
Корзина запроса: [101, 106, 111, -4]
Нашли: False сравнений: 4


---

## Задача 4. Выгодно ли заранее строить индекс?

Для одного запроса можно пройти обычный список. При сотнях запросов
есть смысл один раз подготовить данные. Сравним три готовых способа
проверить **только наличие кода**:

| Способ | Подготовка | Что делает запрос |
|---|---|---|
| `list` | Исходный список уже есть | Просматривает элементы |
| `bisect` | Создаём отсортированную копию | Делит диапазон пополам |
| `set` | Создаём множество уникальных кодов | Использует хеш-индекс |

`set` не хранит названия и число повторов. Если нужны названия, подходит
`dict`, как в первом примере. Бинарный поиск работает на упорядоченных
данных; `bisect_left` даёт место вставки, поэтому нужно ещё проверить
границу и равенство. Место вставки само по себе не означает находку.

Стенд ниже готов. Сначала запишите прогноз: что выберете для одного
отсутствующего запроса и для 200 запросов к неизменной базе?
В каждом случае сравниваем **подготовку вместе с запросами**.

### Ответ (задача 4). Прогноз до замера

Для одного отсутствующего запроса предполагаю, что просмотр list окажется дешевле построения индекса.
Для 200 запросов к неизменной базе предполагаю победу set по общему времени подготовки и поиска.
Это прогноз для данного стенда: конкретные константы определяются запуском.

In [11]:
print("Размер базы:", experiment_n)

Размер базы: 2000


В следующей ячейке все методы получают одинаковые данные и запросы.
Генерация данных не измеряется. При Q=1 есть один промах; при Q=200 —
100 попаданий и 100 промахов. Считается реальный пакет, без пересчёта
одного удачного запроса на целый день. Код можно выполнить как есть.

In [12]:
def binary_contains(data, target):
    index = bisect_left(data, target)
    return index < len(data) and data[index] == target


def lookup_batch(index, queries, contains):
    return [contains(index, target) for target in queries]


def seconds(action):
    # Медиана 5 повторов; каждый повтор содержит 5 одинаковых запусков.
    action()  # один прогрев, вне измерения
    return median(repeat(action, number=BENCHMARK_NUMBER, repeat=BENCHMARK_REPEATS)) / BENCHMARK_NUMBER


def prepare_and_lookup(
    prepare: Callable[[list[int]], Any], data: list[int], queries: list[int], contains: Callable[[Any, int], bool]
) -> list[bool]:
    return lookup_batch(prepare(data), queries, contains)


def search_experiment(n):
    assert n in [1000, 2000, 4000], "Выберите n из списка задания"
    rng = Random(42)
    data = list(range(n))
    rng.shuffle(data)
    strategies = [
        ("list", lambda values: values, lambda values, x: x in values),
        ("bisect", sorted, binary_contains),
        ("set", set, lambda values, x: x in values),
    ]
    results = []
    for q in [1, 200]:
        # Q=1 — один промах; Q=200 — 100 попаданий и 100 промахов.
        queries = [-1] if q == 1 else [rng.randrange(n) for _ in range(100)] + [n + i for i in range(100)]
        rng.shuffle(queries)
        expected = [target in data for target in queries]
        print("Q =", q, "попаданий =", sum(expected))
        print("метод    подготовка мкс   запросы мкс   всё мкс")
        for name, prepare, contains in strategies:
            index = prepare(data)
            assert lookup_batch(index, queries, contains) == expected
            prep = seconds(partial(prepare, data))
            query = seconds(partial(lookup_batch, index, queries, contains))
            total = seconds(partial(prepare_and_lookup, prepare, data, queries, contains))
            results.append((q, name, prep, query, total))
            print(f"{name:7} {prep * 1e6:14.2f} {query * 1e6:13.2f} {total * 1e6:9.2f}")
    return results


if experiment_n is None:
    search_results = None
    print("Эксперимент: НЕ ВЫПОЛНЕНО — выберите experiment_n.")
else:
    search_results = search_experiment(experiment_n)

Q = 1 попаданий = 0
метод    подготовка мкс   запросы мкс   всё мкс
list              0.09         29.31     29.44
bisect          227.30          0.37    227.65
set              42.79          0.21     41.21
Q = 200 попаданий = 100
метод    подготовка мкс   запросы мкс   всё мкс


list              0.08       3916.24   3937.23


bisect          208.84         44.05    257.67
set              37.85         11.20     49.33


Столбец «всё» измерен отдельно. Его число не обязано точно совпасть
с суммой двух других медиан. Время — медиана 5 повторов по 5 запусков;
микросекунды зависят от компьютера и нагрузки. Близкие результаты
стоит перепроверить, а не объявлять гарантированным преимуществом.

Модель роста объясняет наблюдения: при n записях и Q запросах
линейный поиск имеет худшую стоимость O(Qn), сортировка с бинарным
поиском — O(n log n + Q log n) при n≥2, множество — ожидаемо O(n+Q)
при благоприятном хешировании. При плохих коллизиях построение множества
может стать O(n²), а запрос — O(n). Две подготовленные структуры требуют
дополнительной памяти O(n). Секунды одного опыта не доказывают эти оценки.

Если индекс уже существует, не нужно платить за его создание заново.
Если база часто меняется, нужно учитывать обновления. Если требуется
вывести коды диапазона по порядку, одного множества недостаточно.

### Что показать в результате

Сохраните рабочую функцию, собственный каталог с запросом и его выводом.
Добавьте **три коротких вывода**:

1. Что заставило ваш запрос сделать именно столько сравнений?
2. Кто выиграл по полному циклу при Q=1 и Q=200 в вашем запуске?
3. Какую структуру выберете для своего сценария: что храните,
   сколько примерно запросов, нужны ли названия/повторы/порядок?
   Назовите одно ограничение выбора.

### Ответ (задача 4)

1. В собственном каталоге запрос 116 потребовал четыре сравнения: его корзина содержит четыре других кода,
   а совпадения нет, поэтому пришлось дойти до конца цепочки.
2. По полному времени при Q=1 выиграл list, при Q=200 — set; оба прогноза подтвердились.
   В сравнении учтена подготовка индекса, а не только поиск в уже готовой структуре.
3. Для часто проверяемого неизменного набора кодов выбираю set; если нужно получать названия деталей — dict.
   Для единственного запроса достаточно исходного списка. Для диапазонов и вывода в порядке кодов
   подходит отсортированный массив с bisect. Ограничение set — дополнительная память и отсутствие
   упорядоченных диапазонных запросов; построение и обновление индекса тоже требуют работы.

При хороших хешах подготовка set ожидаемо O(n), Q запросов — O(Q); это не гарантия худшего случая.
Как и в ДЗ №2, фильтр Блума имеет смысл как предварительный отсев промахов перед дорогой точной проверкой,
а не как замена точного ответа о наличии. Для маленького множества в памяти он здесь не требуется.

Полное время подготовки и запросов в сохранённом запуске:

| Запросов | list, мкс | bisect, мкс | set, мкс |
| --- | --- | --- | --- |
| 1 | 29.44 | 227.65 | 41.21 |
| 200 | 3937.23 | 257.67 | 49.33 |

При Q=1 запрос отсутствует; при Q=200 половина запросов успешна. Это разные профили нагрузки,
поэтому таблица показывает результат заданных сценариев, а не универсальную точку окупаемости.
Медиана полного времени измерена отдельно и не обязана точно равняться сумме медиан двух этапов.

---

## Задача 5 (со звёздочкой). Удаление кода

По желанию добавьте удаление кода
из цепочки. Проверьте первый, средний, последний и отсутствующий ключ.
После удаления остальные ключи должны по-прежнему находиться.

[Описание bisect](https://docs.python.org/3/library/bisect.html),
[измерений timeit](https://docs.python.org/3/library/timeit.html).

In [13]:
def remove_code(table: ChainedIntSet, target: int) -> bool:
    bucket = table.buckets[target % len(table.buckets)]
    for index, candidate in enumerate(bucket):
        if candidate == target:
            del bucket[index]
            return True
    return False


def test_remove_code() -> None:
    initial = [1, 6, 11, 2, -4]
    for target in (1, 6, -4, 11, 2, 16):
        table = ChainedIntSet(5)
        for value in initial:
            table.add(value)
        before = [bucket[:] for bucket in table.buckets]
        removed = remove_code(table, target)
        assert removed is (target in initial)
        assert {value for bucket in table.buckets for value in bucket} == set(initial) - {target}
        for address, bucket in enumerate(table.buckets):
            expected = [value for value in before[address] if value != target]
            assert bucket == expected
        assert remove_code(table, target) is False
        for value in set(initial) - {target}:
            assert contains_count(table, value)[0]
    assert remove_code(ChainedIntSet(5), 1) is False
    for capacity in (1, 5, 17):
        table = ChainedIntSet(capacity)
        remaining = set(range(-10, 11))
        for value in sorted(remaining):
            table.add(value)
        for target in range(-12, 13):
            assert remove_code(table, target) is (target in remaining)
            remaining.discard(target)
            assert {value for bucket in table.buckets for value in bucket} == remaining
            assert all(contains_count(table, value)[0] for value in remaining)


test_remove_code()
removal_demo = ChainedIntSet(5)
for code in own_codes:
    removal_demo.add(code)
print("До удаления:", removal_demo.buckets)
print("Удалён 106:", remove_code(removal_demo, 106))
print("После удаления:", removal_demo.buckets)
print("Проверки удаления: ПРОЙДЕНО")

До удаления: [[], [101, 106, 111, -4], [], [], [204]]
Удалён 106: True
После удаления: [[], [101, 111, -4], [], [], [204]]
Проверки удаления: ПРОЙДЕНО


### Ответ (задача 5)

Удаление просматривает только корзину target % capacity и удаляет найденный элемент по индексу.
При успехе возвращает True, при отсутствии — False; повторное удаление также возвращает False.
Проверены первый, средний и последний элементы цепочки, отсутствующий ключ, пустая таблица,
отрицательные ключи и последовательное удаление при разном числе корзин.
Внутри корзины используется Python list: поиск и сдвиг после удаления дают O(k) времени для k элементов.
Другие корзины и порядок оставшихся ключей сохраняются; дополнительная память алгоритма O(1).


---

## Сдача

- В первой ячейке указаны ваши фамилия и группа.
- Заполнены ответы к задачам, код функции и данные для собственного примера.
- Ядро перезапущено, ячейки выполнены сверху вниз. Выводы с вашими результатами сохранены в файле.
- Сохраните и сдайте заполненный файл в формате `.ipynb`.

Если проверка не проходит, рядом с ней укажите входные данные, ожидаемый результат и фактический вывод.